In [6]:
#Q1 - a

import numpy as np

# Load data
D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)

x1 = D[:, 0]
y1 = D[:, 3]

# Stack points
points = np.column_stack((x1, y1))

# Compute centroid
centroid = np.mean(points, axis=0)

# Subtract mean
Xc = points - centroid

# SVD
U, S, Vt = np.linalg.svd(Xc)

# Line direction
direction = Vt[0]

# Line normal
normal = Vt[1]

# Line equation: ax + by + c = 0
a, b = normal
c = -a*centroid[0] - b*centroid[1]

print("TLS Line parameters:", a, b, c)

TLS Line parameters: 0.7735616496467872 -0.6337210539312553 -3.794192210845812


In [7]:
#Q1 - b

import random

X_cols = D[:, :3]
Y_cols = D[:, 3:]

X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

points = np.column_stack((X_all, Y_all))

def fit_line(p1, p2):
    a = p2[1] - p1[1]
    b = p1[0] - p2[0]
    c = -(a*p1[0] + b*p1[1])
    return a, b, c

def distance(a, b, c, point):
    x, y = point
    return abs(a*x + b*y + c) / np.sqrt(a*a + b*b)

def ransac(points, iterations=1000, threshold=0.5):
    best_inliers = []

    for _ in range(iterations):
        p1, p2 = random.sample(list(points), 2)
        a, b, c = fit_line(p1, p2)

        inliers = []
        for p in points:
            if distance(a, b, c, p) < threshold:
                inliers.append(p)

        if len(inliers) > len(best_inliers):
            best_inliers = inliers

    return np.array(best_inliers)

# Find 3 lines
remaining = points.copy()
lines = []

for i in range(3):
    inliers = ransac(remaining)
    lines.append(inliers)

    # remove inliers
    remaining = np.array([p for p in remaining if not any((p == inliers).all(1))])